# 官方季度EPS完整请求范围核验
覆盖FY2011Q4至FY2027Q2全部63季度；不把HTTP200新闻目录当公告，不混季度/全年或GAAP/Non-GAAP。缺口保留，原始报告当前副本不获首版/精确公开时钟资格。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.data.nvda_seasonal_eps import extract_quarter_eps
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/nvda_eps_corpus_20261015';p=json.loads((card/'plan.json').read_text());path=root/p['source_manifest'];assert hashlib.sha256(path.read_bytes()).hexdigest()==p['source_manifest_sha256'];m=json.loads(path.read_text());assert len(m['entries'])==p['expected_quarters'];events=[];coverage=[];payloads={};ordinals=[]
for entry in m['entries']:
 for receipt in [entry]+([entry['resolved']] if 'resolved' in entry else []):
  if 'sha256' in receipt:
   data=(path.parent/receipt['path']).read_bytes();assert hashlib.sha256(data).hexdigest()==receipt['sha256'];payloads[receipt['path']]=data
 receipt=entry.get('resolved',entry);row={'fiscal_quarter':entry['expected_quarter'],'source_url':receipt['url'],'http_status':receipt.get('status'),'status':'needs_source','reason':receipt.get('error','')}
 if receipt.get('status')==200:
  try:
   event=extract_quarter_eps(payloads[receipt['path']].decode(),fiscal_year=entry['fiscal_year'],quarter=entry['quarter']);assert pd.Timestamp(event['release_date'])<=pd.Timestamp(m['as_of']);event.update(source_url=receipt['url'],source_sha256=receipt['sha256']);events.append(event);ordinals.append(entry['fiscal_year']*4+entry['quarter']-1);row.update(status='parsed_reference_only',reason='',release_date=event['release_date'],headline_crosschecked=event['headline_crosschecked'],early_release_disclosure=event['early_release_disclosure'])
  except ValueError as error:row.update(status='not_report_document' if 'unexpected release title' in str(error) else 'needs_format_review',reason=str(error))
 coverage.append(row)
assert len({e['fiscal_quarter'] for e in events})==len(events)
longest=[];current=[]
for value in sorted(ordinals):
 if current and value!=current[-1]+1:current=[]
 current.append(value)
 if len(current)>len(longest):longest=current.copy()
def quarter(value):return f'FY{value//4}Q{value%4+1}'
frame=pd.DataFrame([{'datetime':pd.Timestamp(e['release_date']),'instrument':'NVDA_EPS_EVENT','reported_gaap_diluted_eps':float(e['current_eps']),'reported_prior_year_comparative_eps':float(e['prior_year_comparative_eps']),'training_input_qualified':0.,'headline_crosschecked':float(e['headline_crosschecked'])} for e in events]).sort_values('datetime');assert not frame.datetime.duplicated().any();provider=root/'data/qlib/nvda_eps_corpus_20261015_v1';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';fields=['reported_gaap_diluted_eps','reported_prior_year_comparative_eps','training_input_qualified','headline_crosschecked'];manifest=build_qlib_store_from_frame(frame,provider_uri=provider,as_of=m['as_of'],source='Issuer quarterly GAAP diluted EPS current HTML copies; unqualified for training pending vintage and timing audit',field_map={f:f for f in fields});qlib.init(provider_uri=str(provider),region='us');quotes=load_qlib_feature_frame(QlibProviderConfig(provider),instruments=['NVDA_EPS_EVENT'],fields=['$'+f for f in fields],start=str(frame.datetime.min().date()),end=m['as_of']);np.testing.assert_allclose(quotes['$reported_gaap_diluted_eps'],frame.reported_gaap_diluted_eps.to_numpy(),rtol=0,atol=1e-6)
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/raw_html','inputs/source_manifest','analysis/events','analysis/quotes']}
kwargs={'experiment_name':card.name,'uri':str(card/'qlib_recorder')}
if (card/'summary.json').exists():kwargs.update(recorder_id=json.loads((card/'summary.json').read_text())['recorder_id'],resume=True)
with R.start(**kwargs):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_manifest=m,raw_html=payloads,parser=(root/'src/data/nvda_seasonal_eps.py').read_text());rec.save_objects(artifact_path='analysis',events=events,quotes=quotes,provider_manifest=manifest,coverage=pd.DataFrame(coverage));rec.log_metrics(expected_quarters=63,parsed_quarters=len(events),longest_consecutive_quarters=len(longest),registered_artifact_count=8)
summary={'experiment':card.name,'decision':'needs_missing_quarters_split_ledger_and_vintage_clock','new_fits':0,'new_backtests':0,'expected_quarters':63,'parsed_quarters':len(events),'unparsed_quarters':63-len(events),'headline_crosschecked_quarters':sum(e['headline_crosschecked'] for e in events),'early_release_disclosures':sum(e['early_release_disclosure'] for e in events),'longest_consecutive_quarters':len(longest),'longest_run_start':quarter(longest[0]),'longest_run_end':quarter(longest[-1]),'training_input_qualified':False,'full_sue_series_built':False,'provider_uri':str(provider),'region':'us','as_of':m['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope'],'objective_achieved':False}
pd.DataFrame(coverage).to_csv(card/'quarter_coverage.csv',index=False);pd.DataFrame(events).to_csv(card/'source_events.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));binding=json.loads((root/'research/seasonal_eps_source_pilot_20261015/nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=p['source_manifest_sha256'],raw_authority='Official issuer current historical-report copies; quarter identity checked, not certified first vintage');(card/'nas_binding.json').write_text(json.dumps(binding,indent=2));a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2));summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print({k:summary[k] for k in ['parsed_quarters','unparsed_quarters','headline_crosschecked_quarters','early_release_disclosures','longest_consecutive_quarters','longest_run_start','longest_run_end']})


# 原始季度EPS语料扩充：49季度参考值，尚未取得训练资格

本轮0拟合、0回测，按既有事件范围固定检查FY2011Q4至FY2027Q2全部63季度，没有依据事件收益选择公司或日期。候选URL按已观察发行人路径构造，逐份核验最终页面身份；部分HTTP200实际落到News Archive，不当作财报或零EPS。另通过官方搜索结果定位两条真实历史URL，保留最初请求与后续解析的完整来源记录。

取得49季度GAAP diluted EPS结构化参考值，37份还与明确的季度正文EPS交叉一致；另外12份有明确季度GAAP表，但当前模式未找到可作交叉检查的正文句式，未冒充正文已核对。14季度缺口分为13个非报告页面和1个旧表格格式待复核。最长完整段为FY2015Q3至FY2027Q2，共48个连续季度；缺口没有删除出请求清单或填充。

解析器区分季度与全年表，Q4只接受Q4期间列及季度正文，不误取全年EPS；同一table内GAAP/Non-GAAP分段按明确标记停止，不将Non-GAAP行混入。对拆开的Q/FY标记只规范空白，不改数字。泛称Earnings per share的表行要求同时存在明确GAAP diluted正文。原始显示精度保留，全部事件仅登记为Qlib参考观测，training_input_qualified=0，没有前向填充为日频信号。

一个重要时点例外：[FY2015Q1官方公告](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-first-quarter-fiscal-2015)说明因草稿误发而提前在开盘前向市场发布。不能把全部财报统一设为盘后，也不能从其他公告未说明异常就推断其具体时刻。此次历史存档索引探查发生读取超时，仅记录为网络失败，不证明没有历史捕获；没有修改超时结果为“无资料”。

本轮扩充的是发行人历史公告当前HTML副本，而非证明未被修订的首版。仍需将原报值、当期比较值、已知拆股及其他会计重述分开建立账本，逐事件核对发布时间和可用上界。原始Q2拆股原型的两对正确换算不能自动认证所有季度。当前未计算完整SUE、标准化尺度、跨公司排名或ETF权重合成，也不将季节性预测误差叫作分析师一致预期。

49季度不是49×持有天数个独立基本面样本；单一NVDA也不能代表完整历史SOXX成分。即使未来取得输入资格，后续收益实验仍必须固定指标、保留同样本对照、正确处理事件相关性及独立验证，不能由语料覆盖增加推出超额。

决策needs_missing_quarters_split_ledger_and_vintage_clock。3项聚焦解析合同测试、实际语料身份检查、Qlib原生读回及NAS冷恢复逐对象哈希通过。原文和全部请求保留Recorder，季度覆盖/事件小表在卡根目录。生产不变，稳定净SOXX超额目标未完成。
